# 02 - Mainnet CPMM

Raydium CPMM analysis. This notebook is for snapshot-derived CPMM scenarios now and historical counterfactual CPMM swaps once decoded transaction candidates exist.

Inputs:
- `results/real_pool_comparison.csv`
- optional `results/historical_cpmm_candidates.csv`


## Load data


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "helpers").exists():
        sys.path.insert(0, str(candidate))
        break
    if (candidate / "notebooks" / "helpers").exists():
        sys.path.insert(0, str(candidate / "notebooks"))
        break

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from helpers import (
    best_conditions,
    blocker_table,
    data_readiness,
    find_repo_root,
    historical_counterfactual_summary,
    hypothesis_scorecard,
    load_inputs,
    plot_realized_heatmap,
    plot_sensitivity_lines,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

ROOT = find_repo_root()
inputs = load_inputs(ROOT)

comparison = inputs["frames"]["real_pool_comparison"]
historical = inputs["frames"]["historical_cpmm"]

display(data_readiness(ROOT, inputs, ["real_pool_comparison", "historical_cpmm"]))
if comparison.attrs.get("legacy_schema", False):
    display(Markdown("> CPMM comparison CSV is old-schema. Regenerate before final thesis numbers."))


## Snapshot CPMM scorecard


In [ ]:
if comparison.empty:
    display(Markdown("Generate `results/real_pool_comparison.csv` first."))
else:
    display(hypothesis_scorecard(comparison))
    display(blocker_table(comparison))


## Synthetic matched state vs Raydium snapshot


In [ ]:
if comparison.empty:
    display(Markdown("No comparison data."))
else:
    summary = (
        comparison.groupby(["source", "pool_label", "strategy"], dropna=False)
        .agg(
            rows=("source", "size"),
            realized_rate=("attack_realized", "mean"),
            profitable_rate=("attack_profitable", "mean"),
            feasible_rate=("attack_feasible", "mean"),
            median_net_profit=("attacker_net_profit", "median"),
            median_victim_loss_bps=("victim_loss_bps_of_fair_out", "median"),
        )
        .reset_index()
    )
    display(summary)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=comparison, x="source", y="attack_realized", hue="strategy", errorbar=None, ax=axes[0])
    axes[0].set_title("Realized attack rate")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("profitable + feasible rate")
    axes[0].tick_params(axis="x", rotation=15)

    sns.scatterplot(data=comparison, x="victim_size_bps_of_reserve", y="attacker_net_profit", hue="source", style="strategy", ax=axes[1])
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title("Net profit by victim size")
    axes[1].set_xlabel("victim size [bps of reserve_in]")
    axes[1].set_ylabel("attacker net profit")
    plt.tight_layout()


## Historical CPMM candidates


In [ ]:
if historical.empty:
    display(Markdown(
        "No historical CPMM candidates yet. This is the main missing empirical step: collect swaps, decode amount/min_out, reconstruct pre-state, then run `evaluate_historical_cpmm`."
    ))
else:
    display(historical_counterfactual_summary(historical))
    profit_col = "attacker_net_profit" if "attacker_net_profit" in historical else "net_profit"
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(data=historical, x=profit_col, bins=40, element="step", ax=ax)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title("Historical CPMM counterfactual net profit")
    ax.set_xlabel("net profit [token_in units]")
    plt.tight_layout()


## Thesis-ready takeaways


In [ ]:
lines = []
if not comparison.empty:
    lines.append(f"- Snapshot comparison rows: `{len(comparison)}`.")
    lines.append(f"- Snapshot-derived realized rate: `{comparison['attack_realized'].mean():.1%}`.")
if historical.empty:
    lines.append("- Historical CPMM result is not thesis-ready until candidate collection and pre-state reconstruction are done.")
else:
    lines.append(f"- Historical CPMM candidates: `{len(historical)}` rows.")
display(Markdown("\n".join(lines) if lines else "No CPMM conclusions yet."))
